# CAPSTONE 28 — Automatic Topic Tagging for Student Notes

This notebook uses a **synthetic, manually labelled educational-note dataset**. It compares a keyword baseline with a TF-IDF + one-vs-rest logistic-regression multi-label classifier. Tags are suggestions; users remain in control.

## 1. Setup and load data
Run in Google Colab or Jupyter. Required tools: Python, pandas, scikit-learn, matplotlib, and optionally Streamlit for the interface.

In [ ]:
import pandas as pd\nimport numpy as np\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import MultiLabelBinarizer\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.multiclass import OneVsRestClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.metrics import precision_score, recall_score, f1_score, classification_report\n\ndf = pd.read_csv('synthetic_student_notes.csv')\ndf['tags'] = df['tags_string'].fillna('').apply(lambda s: [x for x in s.split(';') if x])\ndf.head()

## 2. Inspect and preprocess
The vectorizer lowercases text and strips accents while preserving meaningful technical tokens such as `C++`, `TF-IDF`, `HTTP`, and code-like words. We keep headings and note text rather than aggressively removing stop words.

In [ ]:
print('Rows:', len(df))\nprint('Empty notes:', df['note_text'].fillna('').str.strip().eq('').sum())\nprint('Tag counts:')\nprint(pd.Series([tag for tags in df['tags'] for tag in tags]).value_counts())

## 3. Multi-label model and hold-out evaluation
A note may have more than one tag. Split the dataset before fitting the vectorizer to avoid test-set leakage.

In [ ]:
taxonomy = pd.read_csv('topic_taxonomy.csv')\nmlb = MultiLabelBinarizer(classes=taxonomy['tag'].tolist())\nY = mlb.fit_transform(df['tags'])\nX_train, X_test, y_train, y_test = train_test_split(df['note_text'], Y, test_size=0.25, random_state=42)\nmodel = Pipeline([('tfidf', TfidfVectorizer(lowercase=True, strip_accents='unicode', ngram_range=(1,2), sublinear_tf=True, token_pattern=r'(?u)\\b[a-zA-Z][a-zA-Z0-9_+#.-]*\\b')), ('clf', OneVsRestClassifier(LogisticRegression(max_iter=2000, class_weight='balanced')))])\nmodel.fit(X_train, y_train)\npred = model.predict(X_test)\nfor avg in ['micro', 'macro']:\n    print(avg, 'precision=', precision_score(y_test,pred,average=avg,zero_division=0), 'recall=', recall_score(y_test,pred,average=avg,zero_division=0), 'F1=', f1_score(y_test,pred,average=avg,zero_division=0))\nprint('Exact match:', (pred == y_test).all(axis=1).mean())

## 4. Keyword baseline
The baseline is intentionally transparent and easy to edit. Rules can miss synonyms and may fire on ambiguous terms.

In [ ]:
RULES = {\n 'Programming':['python','variable','loop','function','code','compiler','dictionary','indentation'],\n 'Algorithms':['algorithm','binary search','merge sort','big o','recursion','complexity','base case'],\n 'Data Structures':['stack','queue','linked list','hash table','array','node','pointer','list'],\n 'Databases':['database','sql','table','primary key','foreign key','join','normalization','transaction'],\n 'Machine Learning':['machine learning','model','training set','test set','overfitting','classifier','neural network','cross-validation','weights'],\n 'Statistics':['precision','recall','mean','median','probability','standard deviation','distribution','confusion matrix','outlier'],\n 'Web Development':['html','css','javascript','http','api','responsive','endpoint','https','media query'],\n 'Software Engineering':['unit test','integration test','git','commit','code review','requirement','compiler','testable'],\n 'Cybersecurity':['encryption','phishing','authentication','least privilege','password','https','credentials','permissions'],\n 'Artificial Intelligence':['artificial intelligence','neural network','natural language processing','generative ai','ai system'],\n 'Information Retrieval':['search engine','tf-idf','cosine similarity','index','rank','vector representation','retrieval'],\n 'Data Analysis':['dataframe','pandas','csv','dashboard','bar chart','line chart','missing values','dataset','filtering','grouping'],\n 'Responsible AI':['privacy','biased','bias','copyright','licence','license','personal identifiers','human oversight','limitations','topic tag','model card']\n}\ndef rule_tags(text):\n    text = text.lower()\n    return [tag for tag, terms in RULES.items() if any(term in text for term in terms)]\nbaseline = np.zeros_like(y_test)\nfor i, text in enumerate(X_test):\n    baseline[i] = [int(tag in rule_tags(text)) for tag in mlb.classes_]\nprint('Baseline micro F1:', f1_score(y_test, baseline, average='micro', zero_division=0))

## 5. Predict tags for a new note
The model's default threshold is 0.5. For a real deployment, tune thresholds on validation data and show confidence/uncertainty. If nothing crosses the threshold, present top candidates as suggestions rather than silently assigning a tag.

In [ ]:
def suggest_tags(note, threshold=0.35, top_k=4):\n    scores = model.decision_function([note])[0]\n    order = np.argsort(scores)[::-1]\n    chosen = [mlb.classes_[i] for i in order if scores[i] > 0][:top_k]\n    return chosen if chosen else [mlb.classes_[i] for i in order[:min(2, top_k)]]\nprint(suggest_tags('A queue uses FIFO order and dequeue removes an item from the front.'))

## 6. Error analysis and limitations
Inspect `test_predictions_error_analysis.csv`. Common issues include overlapping tags (Programming vs Algorithms), short notes with little context, vocabulary absent from the small training set, and code-heavy notes. Results on this small synthetic dataset are a reproducibility demonstration, not evidence of production readiness. Collect consented, representative real notes and use a larger independent test set before deployment.

## 7. Responsible use checklist
- Notes are synthetic in this submission; do not upload private student records without permission.
- Collect only necessary text; remove names, IDs, grades, and other identifiers.
- Tags are suggestions and can be added, removed, or corrected by the student.
- Keep a record of dataset source, licence/permission, split seed, and package versions.
- Review errors across subjects and writing styles; do not use tags for grading or high-stakes decisions.